# NightClear — Full Training Pipeline

Train **NightClear** (DarkIR-based joint low-light enhancement + deblurring) from scratch on **LOL-v2 (Real + Synthetic)**, end to end:

1. Setup & configuration
2. Data: pairing, augmentation, loaders, sanity check
3. Model
4. Loss: `L = L1 + 0.01·LPIPS + 50·Edge + LowRes`
5. Optimizer, scheduler, resume
6. Training loop (mixed precision, checkpoints every epoch)
7. Training curves
8. Final evaluation (PSNR / SSIM / LPIPS)
9. Visual results
10. Export weights for the web app

**Requirements:** a CUDA GPU, the `venv_DarkIR` environment as the kernel (`pip install ipykernel` once), dataset at `data/datasets/LOL-v2`. The notebook can be started from `notebooks/` or the repo root.

## 0. Setup

In [ ]:
# Extra packages used by this notebook and the repo modules it imports (skip if already installed).
# Restart the kernel after the first install if an import still fails.
%pip install -q lpips kornia ptflops einops pytorch_msssim

In [ ]:
import os, sys, json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import lpips
import kornia

# make the repo importable whether the notebook is started from notebooks/ or the repo root
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from archs import DarkIR          # NightClear network (DarkIR architecture)
from losses.loss import EdgeLoss  # Laplacian-pyramid edge loss from the repo

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Repo root :', ROOT)
print('Device    :', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(no GPU: training will be very slow)')
print('PyTorch   :', torch.__version__)

## 1. Configuration

All knobs in one place. The paper trains with batch 32 on an H100; batch 8 at 384×384 crops fits a ~12 GB GPU. If you run out of memory, lower `batch_size` first, then `crop` (keep it a multiple of 8).

In [ ]:
CFG = dict(
    # data
    data_root   = 'data/datasets/LOL-v2',
    subsets     = ['Real_captured', 'Synthetic'],
    crop        = 384,
    batch_size  = 8,
    num_workers = 4,
    # optimisation
    epochs       = 300,
    lr           = 5e-4,
    eta_min      = 1e-6,
    weight_decay = 1e-4,
    betas        = (0.9, 0.999),
    amp          = True,   # mixed precision; the FFT blocks already run in fp32 internally
    # loss weights: L = L1 + 0.01*LPIPS + 50*Edge + LowRes
    w_l1 = 1.0, w_lpips = 0.01, w_edge = 50.0, w_lowres = 1.0,
    # network (NightClear-m: width 32, ~3.3 M params)
    net = dict(img_channel=3, width=32, middle_blk_num_enc=2, middle_blk_num_dec=2,
               enc_blk_nums=[1, 2, 3], dec_blk_nums=[3, 1, 1],
               dilations=[1, 4, 9], extra_depth_wise=True),
    # bookkeeping
    eval_every  = 5,       # epochs between test-set evaluations
    seed        = 42,
    run_dir     = 'models/nightclear_run',
    export_path = 'models/NightClear.pt',
    resume      = True,    # continue from run_dir/last.pt if it exists
)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG['seed'])
torch.backends.cudnn.benchmark = True
RUN_DIR = Path(CFG['run_dir'])
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'config.json').write_text(json.dumps(CFG, indent=2))

## 2. Data

Layout: `LOL-v2/{Real_captured,Synthetic}/{Train,Test}/{Low,Normal}`.
Real uses `lowXXXXX.png` ↔ `normalXXXXX.png`, Synthetic uses identical names, so files are paired by name with the `low`/`normal` prefix stripped.

In [ ]:
def pair_files(subset, split):
    '''Return sorted [(low_path, normal_path), ...] for one subset and split ("Train" / "Test").'''
    base = Path(CFG['data_root']) / subset / split
    key = lambda p: p.name.removeprefix('low').removeprefix('normal')
    lows    = {key(p): p for p in (base / 'Low').glob('*.png')}
    normals = {key(p): p for p in (base / 'Normal').glob('*.png')}
    assert lows, f'no images found in {base / "Low"}'
    assert lows.keys() == normals.keys(), f'unpaired files in {base}'
    return [(lows[k], normals[k]) for k in sorted(lows)]

train_pairs = [p for s in CFG['subsets'] for p in pair_files(s, 'Train')]
test_pairs  = {s: pair_files(s, 'Test') for s in CFG['subsets']}

pd.DataFrame({s: {'train': len(pair_files(s, 'Train')), 'test': len(test_pairs[s])} for s in CFG['subsets']}).T

In [ ]:
def load_rgb(path):
    '''PNG -> float tensor (3, H, W) in [0, 1].'''
    return torch.from_numpy(np.asarray(Image.open(path).convert('RGB'), dtype=np.float32) / 255.).permute(2, 0, 1)

class PairedDataset(Dataset):
    '''Low/normal pairs. With crop: random crop + flips + 90° rotations (same op on both). Without: full image.'''
    def __init__(self, pairs, crop=None):
        self.pairs, self.crop = pairs, crop

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        low, high = (load_rgb(p) for p in self.pairs[i])
        if self.crop:
            c = self.crop
            _, h, w = low.shape
            if h < c or w < c:  # pad small images so the crop always fits
                pad = (0, max(0, c - w), 0, max(0, c - h))
                low, high = F.pad(low, pad, mode='reflect'), F.pad(high, pad, mode='reflect')
                _, h, w = low.shape
            y, x = random.randint(0, h - c), random.randint(0, w - c)
            low, high = low[:, y:y + c, x:x + c], high[:, y:y + c, x:x + c]
            if random.random() < 0.5:
                low, high = low.flip(-1), high.flip(-1)
            if random.random() < 0.5:
                low, high = low.flip(-2), high.flip(-2)
            k = random.randint(0, 3)
            low, high = torch.rot90(low, k, (1, 2)), torch.rot90(high, k, (1, 2))
        return low, high

train_loader = DataLoader(PairedDataset(train_pairs, CFG['crop']), batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True, drop_last=True,
                          persistent_workers=CFG['num_workers'] > 0)
test_loaders = {s: DataLoader(PairedDataset(p), batch_size=1, num_workers=CFG['num_workers'], pin_memory=True)
                for s, p in test_pairs.items()}
print(f'{len(train_pairs)} training pairs -> {len(train_loader)} iterations / epoch')

Sanity check: one augmented batch. Low-light crops should look almost black, ground truth normally exposed.

In [ ]:
low, high = next(iter(train_loader))
print('batch:', tuple(low.shape), f'| mean brightness low {low.mean():.3f} vs normal {high.mean():.3f}')

n = min(4, len(low))
fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 7), squeeze=False)
for j in range(n):
    axes[0, j].imshow(low[j].permute(1, 2, 0));  axes[0, j].set_title('Low-light')
    axes[1, j].imshow(high[j].permute(1, 2, 0)); axes[1, j].set_title('Ground truth')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout(); plt.show()

## 3. Model

Encoder (Fourier domain) brightens the image, decoder (dilated convolutions) removes blur. With `side_loss=True` the network also returns a **1/8-resolution lit image** predicted at the bottleneck, used by the low-res loss.

In [ ]:
model = DarkIR(**CFG['net']).to(DEVICE)
print(f'NightClear parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M')

with torch.no_grad():
    side, out = model(torch.rand(1, 3, 256, 256, device=DEVICE), side_loss=True)
print('full-res output:', tuple(out.shape), '| low-res side output:', tuple(side.shape))

## 4. Loss

| Term | Weight | Purpose |
|---|---|---|
| L1(output, GT) | 1 | pixel fidelity |
| LPIPS-VGG(output, GT) | 0.01 | perceptual quality |
| Edge (Laplacian pyramid) | 50 | sharp edges → deblurring |
| L1(side output, GT ↓8) | 1 | forces the encoder to fix illumination |

The loss is computed in fp32 (outside autocast) for stability.

In [ ]:
class NightClearLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.lpips = lpips.LPIPS(net='vgg', verbose=False).eval()
        for p in self.lpips.parameters():
            p.requires_grad_(False)
        self.edge = EdgeLoss(rank=DEVICE, loss_weight=1.0, criterion='l2')

    def forward(self, out, side, gt):
        gt_low = F.interpolate(gt, size=side.shape[-2:], mode='area')
        parts = {
            'l1':     F.l1_loss(out, gt),
            'lpips':  self.lpips(out, gt, normalize=True).mean(),  # normalize: [0,1] -> [-1,1]
            'edge':   self.edge(out, gt),
            'lowres': F.l1_loss(side, gt_low),
        }
        total = (CFG['w_l1'] * parts['l1'] + CFG['w_lpips'] * parts['lpips']
                 + CFG['w_edge'] * parts['edge'] + CFG['w_lowres'] * parts['lowres'])
        return total, {k: v.item() for k, v in parts.items()}

criterion = NightClearLoss().to(DEVICE)

## 5. Optimizer, scheduler, resume

AdamW with cosine annealing 5e-4 → 1e-6 over all epochs. If `run_dir/last.pt` exists, training continues where it stopped (model, optimizer, scheduler, AMP scaler and history are restored).

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'], betas=CFG['betas'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=CFG['eta_min'])
scaler = torch.amp.GradScaler('cuda', enabled=CFG['amp'] and DEVICE.type == 'cuda')

start_epoch, best_psnr, history = 0, 0.0, []
last_ckpt = RUN_DIR / 'last.pt'
if CFG['resume'] and last_ckpt.exists():
    ckpt = torch.load(last_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch, best_psnr, history = ckpt['epoch'], ckpt['best_psnr'], ckpt['history']
    print(f'Resumed from epoch {start_epoch} (best mean PSNR {best_psnr:.2f} dB)')
else:
    print('Training from scratch')

## 6. Training

Every epoch saves `last.pt`; every `eval_every` epochs the model is scored on both test sets and `best.pt` keeps the highest mean PSNR. The cell can be interrupted and re-run (after re-running section 5) to resume.

In [ ]:
@torch.no_grad()
def evaluate(loader, lpips_fn=None):
    '''Mean PSNR / SSIM (and LPIPS if lpips_fn is given) over full-resolution test images.'''
    model.eval()
    scores = {'psnr': [], 'ssim': []} | ({'lpips': []} if lpips_fn else {})
    for low, high in loader:
        low, high = low.to(DEVICE), high.to(DEVICE)
        with torch.autocast(DEVICE.type, enabled=scaler.is_enabled()):
            out = model(low)
        out = out.float().clamp(0, 1)
        scores['psnr'].append((10 * torch.log10(1 / F.mse_loss(out, high))).item())
        scores['ssim'].append(kornia.metrics.ssim(out, high, window_size=11).mean().item())
        if lpips_fn:
            scores['lpips'].append(lpips_fn(out, high, normalize=True).item())
    model.train()
    return {k: float(np.mean(v)) for k, v in scores.items()}

def save_checkpoint(path, epoch):
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict(), 'best_psnr': best_psnr, 'history': history,
                'config': CFG}, path)

In [ ]:
model.train()
for epoch in range(start_epoch, CFG['epochs']):
    t0, running = time.time(), {}
    for low, high in tqdm(train_loader, desc=f'Epoch {epoch + 1}/{CFG["epochs"]}', leave=False):
        low, high = low.to(DEVICE, non_blocking=True), high.to(DEVICE, non_blocking=True)
        with torch.autocast(DEVICE.type, enabled=scaler.is_enabled()):
            side, out = model(low, side_loss=True)
        loss, parts = criterion(out.float(), side.float(), high)

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        for k, v in {'loss': loss.item(), **parts}.items():
            running[k] = running.get(k, 0.0) + v

    lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    record = {'epoch': epoch + 1, 'lr': lr, 'time_s': time.time() - t0,
              **{k: v / len(train_loader) for k, v in running.items()}}

    if (epoch + 1) % CFG['eval_every'] == 0 or epoch + 1 == CFG['epochs']:
        for s, loader in test_loaders.items():
            record.update({f'{s}/{k}': v for k, v in evaluate(loader).items()})
        mean_psnr = np.mean([record[f'{s}/psnr'] for s in test_loaders])
        if mean_psnr > best_psnr:
            best_psnr = float(mean_psnr)
            save_checkpoint(RUN_DIR / 'best.pt', epoch + 1)

    history.append(record)
    save_checkpoint(last_ckpt, epoch + 1)
    print(' | '.join(f'{k} {v:.4g}' if isinstance(v, float) else f'{k} {v}' for k, v in record.items()))

print(f'Done. Best mean test PSNR: {best_psnr:.2f} dB')

## 7. Training curves

In [ ]:
ep = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for k in ['loss', 'l1', 'lpips', 'edge', 'lowres']:
    axes[0].plot(ep, [h[k] for h in history], label=k)
axes[0].set_yscale('log'); axes[0].set_title('Training loss (total + unweighted terms)')
axes[0].set_xlabel('epoch'); axes[0].legend()

for ax, m in zip(axes[1:], ['psnr', 'ssim']):
    for s in CFG['subsets']:
        pts = [(h['epoch'], h[f'{s}/{m}']) for h in history if f'{s}/{m}' in h]
        if pts:
            ax.plot(*zip(*pts), marker='o', label=s)
    ax.set_title(f'Test {m.upper()}'); ax.set_xlabel('epoch'); ax.legend()

plt.tight_layout(); plt.savefig(RUN_DIR / 'curves.png', dpi=150); plt.show()

## 8. Final evaluation

Reload the best checkpoint and score it on both test sets. LPIPS here uses the standard AlexNet backbone (lower is better).

In [ ]:
best = torch.load(RUN_DIR / 'best.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(best['model_state_dict'])
print(f"Best checkpoint: epoch {best['epoch']}")

lpips_alex = lpips.LPIPS(net='alex', verbose=False).to(DEVICE).eval()
results = {s: evaluate(loader, lpips_alex) for s, loader in test_loaders.items()}
table = pd.DataFrame(results).T.rename(columns=str.upper).round(4)
table.to_csv(RUN_DIR / 'test_metrics.csv')
table

## 9. Visual results

In [ ]:
@torch.no_grad()
def restore(low):
    model.eval()
    return model(low.unsqueeze(0).to(DEVICE)).clamp(0, 1)[0].cpu()

samples = [p for s in CFG['subsets'] for p in random.sample(test_pairs[s], 2)]
fig, axes = plt.subplots(len(samples), 3, figsize=(13, 3.6 * len(samples)), squeeze=False)
for row, (lp, hp) in zip(axes, samples):
    low, high = load_rgb(lp), load_rgb(hp)
    for ax, img, title in zip(row, [low, restore(low), high], ['Low-light input', 'NightClear', 'Ground truth']):
        ax.imshow(img.permute(1, 2, 0)); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.savefig(RUN_DIR / 'samples.png', dpi=120); plt.show()

## 10. Export for the web app

Saves a plain `state_dict` and reloads it the same way the web app does. To serve it, point `DarkIRBackend.checkpoint` in `model_zoo/__init__.py` to `models/NightClear.pt`.

In [ ]:
torch.save(model.state_dict(), CFG['export_path'])
print('Exported weights ->', CFG['export_path'])

from darkir_core import load_checkpoint_state
check = DarkIR(**CFG['net'])
check.load_state_dict(load_checkpoint_state(CFG['export_path']), strict=True)
print('Reload check OK')